# AI Lakehouse — publish and check two results

Origami Health · synthetic training only. Website: lab2. Planned block: 20 minutes.

Run selected prepared cells; do not provision services in class. Configure the full asset pack and runtime bindings before running. Live AIDP execution and classroom timing remain unverified.


## First-time user guide

**Learning objective:** Publish the prepared Gold snapshot into your pre-created Lakehouse tables and check both claim totals and active plans.

**Before you begin:** Notebook 03 passed. Your facilitator prepared the external catalog, tables and approved write access. This notebook never creates database users or grants.

**Key terms:** An external catalog connects Spark to the prepared Lakehouse target. Reconciliation compares persisted outputs with expected values. Idempotent replay means the same snapshot can be repeated without adding duplicate rows.

**Timebox:** This is the shared 20-minute block for notebooks 03 AND 04, not 20 minutes each. Read the explanations as you run the prepared cells. Optional exploration is not part of the core timebox.

**How to run:** There are 3 code cells below. Run them one at a time, from top to bottom, using the editor's run-cell control. Wait for completion, read the actual output and only then continue. Do not use Run all on your first pass. All runtime bindings are facilitator-prepared; do not paste credentials or edit another participant's paths.

For common problems, open START_HERE.md at the asset-pack root. If an assertion fails, stop and keep the error for the facilitator; do not change the assertion or expected values to make it pass.


## Load administrator-prepared participant bindings

**Purpose:** Load administrator-prepared participant bindings

**Inputs:** Full asset pack plus external ORIGAMI_ASSET_ROOT / ORIGAMI_RUNTIME_CONFIG bindings.

**Processing:** Run the provided bounded code in order.

**Validation:** Assertions must pass; save results and record any fallback.

**Expected output:** Assigned slot and synthetic snapshot; placeholders fail closed.

**Business meaning:** Governed synthetic data supports analysis, not coverage decisions.

**Your action — step 1:** Run the binding cell; confirm the assigned slot before any publication.

**Before moving on:** wait for completion, inspect the actual output against the expected result above, and stop if any error appears.


In [ ]:
import os, sys, json
from pathlib import Path
asset_root = os.environ.get('ORIGAMI_ASSET_ROOT')
if not asset_root:
    raise RuntimeError('Facilitator must configure ORIGAMI_ASSET_ROOT and ORIGAMI_RUNTIME_CONFIG before class.')
sys.path.insert(0, str(Path(asset_root) / 'shared'))
from origami_core import AS_OF, SOURCES, fingerprint, score_rows
from aidp_runtime import configuration, bounded_rows, persist, read_delta, source_frame, verify_fixture, unique, publish_snapshot, save_evidence
cfg = configuration()
from pyspark.sql import functions as F
from pyspark.sql.window import Window
spark.sparkContext.addPyFile(str(Path(asset_root) / 'shared' / 'origami_core.py'))
print('Participant:', cfg['participant_id'], '| synthetic snapshot:', AS_OF)

## Publish into pre-created participant tables

**Purpose:** Publish into pre-created participant tables

**Inputs:** Approved external catalog, pre-created target tables, explicit runtime write gate.

**Processing:** Run the provided bounded code in order.

**Validation:** Assertions must pass; save results and record any fallback.

**Expected output:** Identical snapshot inserted/reconciled; changed existing rows stop execution.

**Business meaning:** Governed synthetic data supports analysis, not coverage decisions.

**Your action — step 2:** Run publication once and wait for the inserted-and-reread message. Identical existing rows are reused. Different or unexpected existing rows deliberately stop the load—ask the facilitator, never drop or truncate a table.

**Before moving on:** wait for completion, inspect the actual output against the expected result above, and stop if any error appears.


In [ ]:
keys={'claim_detail':['claim_id'],'claims_monthly':['service_month','provider_id','service_code'],'member_plan':['enrollment_id'],'benefit_usage':['usage_id'],'benefit_schedule':['benefit_id']}
for name,key in keys.items():
    publish_snapshot(spark,cfg,name,read_delta(spark,cfg,'gold_'+name),key)
print('Snapshot inserted and reread exactly; unchanged sequential reruns add no rows. Concurrent writers and changed snapshots are unsupported.')

## Run the two participant checks

**Purpose:** Run the two participant checks

**Inputs:** Persisted Lakehouse tables and expected fixture manifest.

**Processing:** Run the provided bounded code in order.

**Validation:** Assertions must pass; save results and record any fallback.

**Expected output:** Claim count/amount totals match; assigned synthetic member has BASE and PLUS active plans.

**Business meaning:** Governed synthetic data supports analysis, not coverage decisions.

**Your action — step 3:** Run the two checks. First compare claim counts and centavo totals with the fixture. Then check that your assigned member has BASE and PLUS 2026 plans, not the expired 2025 enrollment.

**Before moving on:** wait for completion, inspect the actual output against the expected result above, and stop if any error appears.


In [ ]:
detail=spark.table(f"{cfg['target_catalog']}.{cfg['target_schema']}.origami_claim_detail")
detail=detail.toDF(*[c.lower() for c in detail.columns])
actual=detail.agg(F.count('*').alias('claim_count'),F.sum('submitted_centavos').alias('submitted_centavos'),F.sum('paid_centavos').alias('paid_centavos')).first().asDict()
expected=json.loads((Path(asset_root)/'contracts'/'expected.json').read_text())
assert {k:int(v) for k,v in actual.items()}==expected['claim_totals']
member=expected['slot_cases'][cfg['participant_id']]['authorized_member']
mp=spark.table(f"{cfg['target_catalog']}.{cfg['target_schema']}.origami_member_plan")
mp=mp.toDF(*[c.lower() for c in mp.columns])
active=mp.where((F.col('member_id')==member)&(F.col('start_date')<=AS_OF)&(F.col('end_date')>=AS_OF)&(F.col('enrollment_status')=='ACTIVE'))
assert sorted(r['plan_id'] for r in active.select('plan_id').collect())==['BASE','PLUS']
active.select('member_id','plan_id','plan_version','document_id').show(truncate=False)
save_evidence(cfg,'lakehouse-two-checks',{'totals':actual,'active_plans':['BASE','PLUS'],'member':member})

## Completion checkpoint

- [ ] Every required code cell completed without error.
- [ ] I checked the actual result: Five target tables reconcile exactly. Claim count is 720 and both monetary totals match contracts/expected.json. The displayed member-plan rows contain BASE and PLUS with 2026 versions.
- [ ] I saved evidence: Save the actual totals and active-plan output, plus lakehouse-two-checks.json. Do not substitute expected-snapshot files for evidence of a live run.

**Discuss:** Does having two active plans mean the Copilot may add their benefit limits together?

**Self-check explanation:** No. Active enrollment establishes two separate plan relationships. Coordination rules are not supplied; the representative must review that unresolved question.

**Next:** day1/governance.md, then the Day 1 catch-up checkpoint. Continue only after the core checkpoint passes, or after the facilitator explicitly records a labelled fallback.

## If you get stuck

Note the notebook name, cell heading and sanitized error. Missing configuration, permission errors and missing target tables are facilitator setup issues, not instructions to provision resources. Unexpected values are validation failures: do not erase tables, disable checks or rerun the entire pipeline blindly. See START_HERE.md for the troubleshooting table.
